In [2]:
from pprint import pprint
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor

print('PyTorch version:', torch.__version__)


PyTorch version: 2.13.0+cpu


In [3]:
"""
手动更新到optimizer.step()
"""

x = torch.randn(8, 3)
y = torch.randn(8, 1)

model = nn.Linear(3, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)

before = model.weight.detach().clone()

pred = model(x)
loss = F.mse_loss(pred, y)

optimizer.zero_grad()
loss.backward()
optimizer.step()

after = model.weight.detach().clone()

flag = torch.allclose(before, after)
max_diff = (before - after).abs().max().item()

print('Is the parameters unchanged:', flag)
print('Max absolute difference:', max_diff)


Is the  parameters  unchanged: False
Max absolute difference: 0.05386516451835632


In [5]:
"""
为什么每次更新前都要zero_grad()
    - PyTorch 中的梯度是默认累加的,连续调用两次backward(),第二次的梯度不会覆盖第一次,而是加到第一次的.grad上
"""
w = torch.tensor([1.0], requires_grad=True)

loss1 = w.pow(2)
loss1.backward()
print('After first backward:', w.grad)

loss2 = w.pow(2)
loss2.backward()
print('After second backward:', w.grad)

# 故意多个mini-batch的梯度,在统一更新一次参数,这种叫做梯度累积

model = nn.Linear(3, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)
optimizer.zero_grad()

for i in range(4):
    x = torch.randn(8, 3)
    y = torch.randn(8, 1)

    pred = model(x)
    loss = F.mse_loss(pred, y)
    loss = loss / 4
    loss.backward()
optimizer.step()


After first backward: tensor([2.])
After second backward: tensor([4.])


In [8]:
"""
set_to_none 是什么
    optimiter.zero_grad() 默认会把参数的.grad 置为None
"""

model = nn.Linear(3, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)

x = torch.randn(8, 3)
y = torch.randn(8, 1)

loss = F.mse_loss(model(x), y)
loss.backward()

flag1 = model.weight.grad is None
print('Whether  grad is None before  zero_grad:', flag1)

optimizer.zero_grad(set_to_none=True)

flag2 = model.weight.grad is None
print('Whether  grad is None after  zero_grad:', flag2)






Whether  grad is None before  zero_grad: False
Whether  grad is None after  zero_grad: True


In [10]:
"""
参数组: 不同的参数可以有不同的学习率
    -  针对不同的参数配置不同的学习率
    - 也可以通过分组实现学习率的不同设定
"""


class TinyModel(nn.Module):
    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
        )

        self.head = nn.Linear(32, 2)

    def forward(self, x: Tensor) -> Tensor:
        x = self.backbone(x)
        return self.head(x)


model = TinyModel()
optimizer = optim.SGD(
    [
        {'params': model.backbone.parameters(), 'lr': 1e-4},
        {'params': model.head.parameters(), 'lr': 1e-3},
    ],
    weight_decay=1e-2,

)

for i, group in enumerate(optimizer.param_groups):
    print(f"Parameter group {i}:lr={group['lr']},weight_decay={group['weight_decay']}")


# 设置某些参数不做weight  decay

def split_weight_decay_params(model: nn.Module):
    decay = []
    no_decay = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.endswith('bias') or 'norm' in name.lower():
            no_decay.append(param)
        else:
            decay.append(param)
    return [
        {'params': no_decay, 'weight_decay': 1e-2},
        {'params': decay, 'weight_decay': 0},
    ]


model = nn.Sequential(
    nn.Linear(10, 32),
    nn.LayerNorm(32),
    nn.Linear(32, 2),
)

optimizer = optim.SGD(split_weight_decay_params(model), weight_decay=1e-3)
for i, group in enumerate(optimizer.param_groups):
    print(
        f"Parameter  group {i}, weight_decay:{group['weight_decay']}, n_params:{group['params']}"
    )

Parameter group 0:lr=0.0001,weight_decay=0.01
Parameter group 1:lr=0.001,weight_decay=0.01
Parameter  group 0, weight_decay:0.01, n_params:[Parameter containing:
tensor([-0.0359, -0.2747, -0.3062,  0.0166,  0.3108, -0.2246, -0.2838, -0.3006,
         0.1389, -0.2414, -0.0673, -0.3143,  0.2292, -0.2279, -0.2007,  0.0550,
         0.1750, -0.0303,  0.2607, -0.2627, -0.1276, -0.0135, -0.1100,  0.0595,
        -0.0879, -0.1639,  0.2714,  0.1980,  0.0424,  0.0476, -0.0168,  0.0766],
       requires_grad=True), Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True), Parameter containing:
tensor([-0.1166,  0.0498], requires_grad=True)]
Parameter  group 1, weight_decay:0, n_params:[Parameter containing:
tensor([[-0.1285, -0.0254,  0.1948, -0.2083,  0.2538,  0.2793,  0.2147, -0.3107,
         -0.0869,  0.2246],
        [ 0.2667, -0.1603, -0.1154,  0.0386, -0.1842, 

In [19]:
"""
优化器不只是公式,也有状态
    - 优化器参数会根据之前的状态做出调整
        1. state:每个参数对应的优化器调整
        2. param_groups:参数配置,比如学习率,weight decay,betas 等
"""

model = nn.Linear(3, 2)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
pprint(optimizer.state_dict(), sort_dicts=False)

# 执行一次更新后
x = torch.randn(8, 3)
y = torch.randn(8, 1)

loss = F.mse_loss(model(x), y)

optimizer.zero_grad()
loss.backward()
optimizer.step()

state_dict = optimizer.state_dict()
pprint(optimizer.state_dict(), sort_dicts=False)

# 保存的断点
checkpoint = {
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
}

torch.save(checkpoint, './checkpoint.pth')

# 加载时加载两部分
model = nn.Linear(3, 2)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

checkpoint = torch.load('./checkpoint.pth')
model.load_state_dict(checkpoint['model'])
optimizer.load_state_dict(checkpoint['optimizer'])


{'state': {},
 'param_groups': [{'lr': 0.001,
                   'betas': (0.9, 0.999),
                   'eps': 1e-08,
                   'weight_decay': 0.01,
                   'amsgrad': False,
                   'maximize': False,
                   'foreach': None,
                   'capturable': False,
                   'differentiable': False,
                   'fused': None,
                   'decoupled_weight_decay': True,
                   'params': [0, 1]}]}
{'state': {0: {'step': tensor(1.),
               'exp_avg': tensor([[ 0.0240,  0.0076,  0.0401],
        [ 0.0460,  0.0148, -0.0439]]),
               'exp_avg_sq': tensor([[5.7662e-05, 5.7356e-06, 1.6047e-04],
        [2.1170e-04, 2.1846e-05, 1.9276e-04]])},
           1: {'step': tensor(1.),
               'exp_avg': tensor([-0.0420, -0.0171]),
               'exp_avg_sq': tensor([1.7680e-04, 2.9229e-05])}},
 'param_groups': [{'lr': 0.001,
                   'betas': (0.9, 0.999),
                   'eps': 1e-0

C:\Users\stone\AppData\Local\Temp\ipykernel_3552\1647212924.py:16: UserWarning: Using a target size (torch.Size([8, 1])) that is different to the input size (torch.Size([8, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(model(x), y)


In [20]:
"""
foreach 和fused: 同一个优化器的不同实现方式
    - for-loop:最传统的方式,逐个参数张量更新
    - foreach: 把一组张量打包,调用批量操作
    - fused: 把多个更新操作融合到更少的kernel中
"""

model = nn.Linear(3, 2)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, foreach=False, fused=False)

pprint(optimizer)


AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: False
    fused: False
    lr: 0.001
    maximize: False
    weight_decay: 0.01
)


In [21]:
"""
optimizer.step() 默认不会记录图
    - 大多数的情况: differentiable =  False就可以,只需要对参数更新过程中额求导的特殊场景下,才考虑differentiable = True
"""
model = nn.Linear(3, 2)
optimizer = optim.SGD(model.parameters(), lr=1e-3, differentiable=True)

flag = optimizer.defaults['differentiable']
print("Is Optimizer  step  differentiable:", flag)

Is Optimizer  step  differentiable: True


In [24]:
"""
一个完整的优化步骤
    model(x):执行了前向传播,计算预测值,并构建图
    loss.backward():根据当前的loss计算梯度,并把结果累加到参数模型的.grad属性上
    optimizer.step(): 根据惨参数的.grad 和优化器的内部状态,更新参数
    optimizer.zero_grad():清空旧梯度,避免下次的backword() 继续累加
"""

model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

optimizer = optim.AdamW([
    {'params': model[0].parameters(), 'lr': 1e-3},
    {'params': model[2].parameters(), 'lr': 1e-3},
],
    weight_decay=1e-2, )

x = torch.randn(16, 10)
y = torch.randn(16, 1)

model.train()
pred = model(x)
loss = F.mse_loss(pred, y)

optimizer.zero_grad()
loss.backward()
optimizer.step()
print('loss:', loss.item())






loss: 1.4036568403244019
